In [77]:
import os
import glob
import numpy as np
import nibabel as nib
import SimpleITK as sitk
from tqdm.notebook import tqdm
import sklearn # Required for train_test_split, ensure it's installed if not already

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.transforms import (
    Compose, Orientationd, Spacingd, CropForegroundd,
    ScaleIntensityRanged, NormalizeIntensityd, ToTensord,
    RandSpatialCropd, RandFlipd, RandGaussianNoised, RandAdjustContrastd,
    RandBiasFieldd, EnsureChannelFirstd
)
from monai.networks.nets import UNet, SegResNet
from monai.data import decollate_batch
from monai.metrics import DiceMetric
from monai.losses import DiceLoss, DiceCELoss

# Conditional import for intensity_normalization
_HAS_INTENSITY_NORMALIZATION = False
try:
    # Attempt import directly from top-level or other common sub-modules if 'normalize' doesn't work
    # Or, specifically target a version if the API changed.
    # The common path is intensity_normalization.normalize
    from intensity_normalization.normalize import nyul_train_standard_scale, nyul_apply_standard_scale
    _HAS_INTENSITY_NORMALIZATION = True
except ImportError as e:
    print(f"Warning: 'intensity-normalization.normalize' module or its specific functions (nyul_train_standard_scale, nyul_apply_standard_scale) not importable: {e}")
    print("Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.")

# Conditional import for itk-elastix (still challenging to install via pip)
_HAS_ITK_ELASTIX = False
try:
    import itk # itk-elastix is often used with itk directly
    _HAS_ITK_ELASTIX = True
except ImportError:
    print("Warning: 'itk-elastix' not directly importable. Robust registration features will be limited.")


# --- Offline Preprocessing Functions (Conceptual / External) ---
# These functions illustrate steps that *should* ideally be run once offline
# to prepare perfectly aligned and normalized data.
# Due to SimpleITK-Elastix installation challenges, these are provided as a conceptual
# guide for manual execution outside this notebook's direct runtime.

def apply_n4_bias_correction(image_path, output_path):
    """
    Applies N4 Bias Field Correction to a NIfTI image.
    NOTE: This is a conceptual function. You should run this offline.
    """
    try:
        input_image = sitk.ReadImage(image_path, sitk.sitkFloat32)
        mask_image = sitk.OtsuThreshold(input_image, 0, 1, 200)

        corrector = sitk.N4BiasFieldCorrectionImageFilter()
        corrector.SetMaximumNumberOfIterations([100, 100, 60, 40])
        corrected_image_sitk = corrector.Execute(input_image, mask_image)
        log_bias_field = corrector.GetLogBiasFieldAsImage(input_image)
        corrected_image_full_resolution = input_image / sitk.Exp(log_bias_field)

        sitk.WriteImage(corrected_image_full_resolution, output_path)
        print(f"N4 corrected and saved: {output_path}")
    except Exception as e:
        print(f"Error applying N4 to {image_path}: {e}")

def apply_nyul_normalization(image_path, output_path, standard_scale):
    """
    Applies Nyul & Udupa normalization to a NIfTI image.
    NOTE: This is a conceptual function. You should run this offline.
    Requires 'intensity-normalization' library.
    """
    if not _HAS_INTENSITY_NORMALIZATION:
        print("Skipping Nyul normalization: 'intensity-normalization' not available or functions not found.")
        return
    try:
        img = nib.load(image_path)
        data = img.get_fdata()
        normalized_data = nyul_apply_standard_scale(data, standard_scale)
        normalized_img = nib.Nifti1Image(normalized_data.astype(np.float32), img.affine, img.header)
        nib.save(normalized_img, output_path)
        print(f"Nyul normalized and saved: {output_path}")
    except Exception as e:
        print(f"Error applying Nyul normalization to {image_path}: {e}")

def register_image_and_mask(fixed_image_path, moving_image_path, moving_mask_path, output_dir, subject_id, modality_name):
    """
    Registers a moving image to a fixed image using SimpleITK.Elastix (if available)
    and applies the same transformation to the corresponding mask.
    NOTE: This is a conceptual function. It's recommended to run robust registration offline.
    """
    if not _HAS_ITK_ELASTIX:
        print(f"Skipping Elastix registration for {modality_name}: 'itk-elastix' not available.")
        return

    try:
        fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
        moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)
        moving_mask = sitk.ReadImage(moving_mask_path, sitk.sitkUInt8)

        parameter_object = sitk.ParameterObject()
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("rigid"))
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("affine"))
        parameter_object.AddParameterMap(parameter_object.GetDefaultParameterMap("bspline"))

        for i in range(parameter_object.GetNumberOfParameterMaps()):
            parameter_object.SetParameter(i, "Metric", "AdvancedMattesMutualInformation")

        elastix_filter = sitk.ElastixImageFilter()
        elastix_filter.SetFixedImage(fixed_image)
        elastix_filter.SetMovingImage(moving_image)
        elastix_filter.SetParameterObject(parameter_object)
        elastix_filter.Execute()

        registered_image = elastix_filter.GetResultImage()
        sitk.WriteImage(registered_image, os.path.join(output_dir, f'{subject_id}_{modality_name}_preprocessed.nii.gz'))

        transform_param_map = elastix_filter.GetTransformParameterMap()
        transformix_filter = sitk.TransformixImageFilter()
        transformix_filter.SetMovingImage(moving_mask)
        transformix_filter.SetTransformParameterMap(transform_param_map)
        transformix_filter.Execute()

        registered_mask = transformix_filter.GetResultImage()
        sitk.WriteImage(registered_mask, os.path.join(output_dir, f'{subject_id}_lesion-msk_registered.nii.gz'))

        print(f"Registered {modality_name} and its mask for {subject_id}")
    except Exception as e:
        print(f"Error during registration for {subject_id} {modality_name}: {e}")

# --- End Offline Preprocessing Functions ---

# --- Modified ISLESDataset3D to load RAW files and apply minimal on-the-fly standardization ---
# This version loads raw files and relies on MONAI transforms for initial spatial alignment
# and intensity standardization. For full "perfect alignment," run Elastix offline.
class ISLESDataset3D(Dataset):
    def __init__(self, raw_data_root_dir, raw_mask_root_dir):
        self.samples = []
        print(f"Scanning for 3D samples in raw data root: {raw_data_root_dir}")

        subject_dirs = sorted(glob.glob(os.path.join(raw_data_root_dir, 'sub-*')))

        for subject_dir in subject_dirs:
            subject_id = os.path.basename(subject_dir)
            ses_dwi_dir = os.path.join(subject_dir, "ses-0001", "dwi")
            ses_anat_dir = os.path.join(subject_dir, "ses-0001", "anat")
            mask_dir = os.path.join(raw_mask_root_dir, subject_id, "ses-0001")

            dwi_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz'))
            adc_files = glob.glob(os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz'))
            flair_files = glob.glob(os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.gz'))
            mask_files = glob.glob(os.path.join(mask_dir, f'{subject_id}_ses-0001_msk.nii.gz'))

            if dwi_files and adc_files and flair_files and mask_files:
                self.samples.append({
                    "image_dwi": dwi_files[0],
                    "image_adc": adc_files[0],
                    "image_flair": flair_files[0],
                    "label": mask_files[0],
                    "subject_id": subject_id
                })
            else:
                print(f"Skipping subject {subject_id} due to missing files:")
                if not dwi_files: print(f"  Missing DWI: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_dwi.nii.gz')}")
                if not adc_files: print(f"  Missing ADC: {os.path.join(ses_dwi_dir, f'{subject_id}_ses-0001_adc.nii.gz')}")
                if not flair_files: print(f"  Missing FLAIR: {os.path.join(ses_anat_dir, f'{subject_id}_ses-0001_FLAIR.nii.')}")
                if not mask_files: print(f"  Missing Mask: {os.path.join(mask_dir, f'{subject_id}_ses-0001_lesion-msk.nii')}")

        print(f"Total 3D samples found: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample_paths = self.samples[idx]

        try:
            # Load images as SimpleITK objects, then convert to NumPy arrays.
            # MONAI transforms will handle further spatial/intensity standardization.
            dwi_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_dwi"], sitk.sitkFloat32)).astype(np.float32)
            adc_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_adc"], sitk.sitkFloat32)).astype(np.float32)
            flair_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["image_flair"], sitk.sitkFloat32)).astype(np.float32)
            mask_data = sitk.GetArrayFromImage(sitk.ReadImage(sample_paths["label"], sitk.sitkUInt8)).astype(np.float32)

            mask_data = (mask_data > 0.5).astype(np.float32)

            image_stacked = np.stack([dwi_data, adc_data, flair_data], axis=0)

            return {"image": image_stacked, "label": mask_data, "subject_id": sample_paths["subject_id"]}

        except Exception as e:
            print(f"Error loading or processing sample {sample_paths.get('subject_id', 'N/A')}: {e}")
            raise

# --- Data Loading and Preprocessing (Step 1: Define Data Paths and Structure) ---

# Adjusting data_root based on your "refer previous directory ../data" instruction.
# This assumes your notebook is in a directory like `/content/project/notebooks/`
# and your ISLES-2022 data is located at `/content/project/data/ISLES-2022/`.

base_project_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
base_data_dir = os.path.join(base_project_dir, 'data')

data_root =base_data_dir
mask_root_for_raw = os.path.join(base_data_dir, 'derivatives')

# This `preprocessed_data_root` is where *you would save* the output of offline preprocessing.
# The `ISLESDataset3D` in this version is *not* reading from here; it's reading from `data_root` (raw data).
preprocessed_data_root = os.path.join(base_data_dir, 'ISLES-2022_preprocessed')

print(f"Base Project Directory: {base_project_dir}")
print(f"Resolved Raw Data Root: {data_root}")
print(f"Resolved Raw Mask Root: {mask_root_for_raw}")
print(f"Conceptual Preprocessed Data Root (for offline saving): {preprocessed_data_root}")

os.makedirs(preprocessed_data_root, exist_ok=True)

# --- Conceptual Offline Preprocessing Execution (You would run this once, externally) ---
# This block is for demonstration only. Do NOT run this unless you intend to
# execute the entire, lengthy preprocessing pipeline externally.
# It requires `intensity-normalization` and potentially `itk-elastix` to be functional.
#
# all_raw_subject_dirs = sorted(glob.glob(os.path.join(data_root, 'sub-*')))
#
# for raw_subject_dir in tqdm(all_raw_subject_dirs, desc="Conceptual Offline Preprocessing"):
#     raw_subject_id = os.path.basename(raw_subject_dir)
#     output_subject_dir = os.path.join(preprocessed_data_root, raw_subject_id, 'ses-0001')
#     os.makedirs(output_subject_dir, exist_ok=True)
#
#     current_raw_dwi_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_dwi.nii.gz')
#     current_raw_adc_path = os.path.join(raw_subject_dir, 'ses-0001', 'dwi', f'{raw_subject_id}_ses-0001_adc.nii.gz')
#     current_raw_flair_path = os.path.join(raw_subject_dir, 'ses-0001', 'anat', f'{raw_subject_id}_ses-0001_FLAIR.nii.gz')
#     current_raw_mask_path = os.path.join(mask_root_for_raw, raw_subject_id, 'ses-0001', f'{raw_subject_id}_ses-0001_lesion-msk.nii.gz')
#
#     if not all(os.path.exists(f) for f in [current_raw_dwi_path, current_raw_adc_path, current_raw_flair_path, current_raw_mask_path]):
#         # print(f"Skipping conceptual preprocessing for {raw_subject_id}: Raw files not found.") # Uncomment for verbose skipping
#         continue
#
#     # N4 Bias Correction
#     n4_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4.nii.gz')
#     n4_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4.nii.gz')
#     n4_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4.nii.gz')
#     # apply_n4_bias_correction(current_raw_dwi_path, n4_dwi_path)
#     # apply_n4_bias_correction(current_raw_adc_path, n4_adc_path)
#     # apply_n4_bias_correction(current_raw_flair_path, n4_flair_path)
#
#     # Nyul & Udupa Normalization (Requires pre-trained scales and _HAS_INTENSITY_NORMALIZATION to be True)
#     # This part assumes you have already run nyul_train_standard_scale on a representative subset
#     # of your N4-corrected training data and saved the scales.
#     # If you run Nyul, replace n4_paths with nyul_paths in registration.
#     #
#     # try:
#     #     dwi_standard_scale = np.load(os.path.join(preprocessed_data_root, 'dwi_nyul_scale.npy'))
#     #     adc_standard_scale = np.load(os.path.join(preprocessed_data_root, 'adc_nyul_scale.npy'))
#     #     flair_standard_scale = np.load(os.path.join(preprocessed_data_root, 'flair_nyul_scale.npy'))
#     #
#     #     nyul_dwi_path = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4_nyul.nii.gz')
#     #     nyul_adc_path = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4_nyul.nii.gz')
#     #     nyul_flair_path = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4_nyul.nii.gz')
#     #
#     #     apply_nyul_normalization(n4_dwi_path, nyul_dwi_path, dwi_standard_scale)
#     #     apply_nyul_normalization(n4_adc_path, nyul_adc_path, adc_standard_scale)
#     #     apply_nyul_normalization(n4_flair_path, nyul_flair_path, flair_standard_scale)
#     # except FileNotFoundError:
#     #     # print(f"Skipping Nyul normalization for {raw_subject_id}: Standard scales not found.") # Uncomment for verbose skipping
#     #     pass # Skip if scales are not pre-trained
#
#     # Multimodal Registration (Elastix)
#     # fixed_image_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_flair_n4_nyul.nii.gz') # Or _n4.nii.gz if Nyul skipped
#     # moving_dwi_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_dwi_n4_nyul.nii.gz') # Or _n4.nii.gz
#     # moving_adc_for_reg = os.path.join(output_subject_dir, f'{raw_subject_id}_adc_n4_nyul.nii.gz') # Or _n4.nii.gz
#     #
#     # register_image_and_mask(fixed_image_for_reg, moving_dwi_for_reg, current_raw_mask_path, output_subject_dir, raw_subject_id, 'dwi')
#     # register_image_and_mask(fixed_image_for_reg, moving_adc_for_reg, current_raw_mask_path, output_subject_dir, raw_subject_id, 'adc')
#
#     # Copy/rename FLAIR as "preprocessed" as it's the fixed reference
#     # shutil.copy(fixed_image_for_reg, os.path.join(output_subject_dir, f'{raw_subject_id}_flair_preprocessed.nii.gz'))
#
# print("--- Conceptual offline preprocessing section. For production, run these steps externally. ---")


# Create full dataset instance using the RAW data root (as Elastix is problematic to install)
# This means MONAI transforms will handle initial spatial standardization (Orientation, Spacing).
# For perfect alignment, you *must* run the full N4+Nyul+Elastix pipeline offline and then
# modify ISLESDataset3D to read from `preprocessed_data_root`.
full_dataset = ISLESDataset3D(data_root, mask_root_for_raw)

# Split indices for training and validation
from sklearn.model_selection import train_test_split
train_indices, val_indices = train_test_split(range(len(full_dataset)), test_size=0.2, random_state=42)

# Create subset datasets
train_ds = torch.utils.data.Subset(full_dataset, train_indices)
val_ds = torch.utils.data.Subset(full_dataset, val_indices)

print(f"Total training subjects: {len(train_ds)}")
print(f"Total validation subjects: {len(val_ds)}")

# --- MONAI Transforms for Data Augmentation and Spatial Standardization ---
# These transforms will be applied to the NumPy arrays returned by ISLESDataset3D.__getitem__
# They handle channel addition, orientation, spacing, and intensity transforms.

target_spacing = (1.0, 1.0, 1.0) # Example target spacing for final data
roi_size = (128, 128, 128) # Example patch size for training

train_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys="image", subtrahend=0.5, divisor=0.5),

        RandSpatialCropd(keys=["image", "label"], roi_size=roi_size, random_size=False, random_center=True),
        RandFlipd(keys=["image", "label"], spatial_axis=0, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=1, prob=0.5),
        RandFlipd(keys=["image", "label"], spatial_axis=2, prob=0.5),
        RandGaussianNoised(keys=["image"], prob=0.1, std=0.01),
        RandAdjustContrastd(keys=["image"], prob=0.1, gamma=(0.7, 1.3)),
        RandBiasFieldd(keys=["image"], prob=0.1, coeff_range=(0.0, 0.05)),

        ToTensord(keys=["image", "label"]),
    ]
)

val_transforms = Compose(
    [
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="RAS"),
        Spacingd(keys=["image", "label"], pixdim=target_spacing, mode=("bilinear", "nearest")),
        CropForegroundd(keys=["image", "label"], source_key="image", k_divisible=roi_size),
        ScaleIntensityRanged(keys="image", a_min=0, a_max=2000, b_min=0.0, b_max=1.0, clip=True),
        NormalizeIntensityd(keys=["image"], subtrahend=0.5, divisor=0.5),
        ToTensord(keys=["image", "label"]),
    ]
)

# Apply MONAI transforms to the subset datasets
train_ds.transform = train_transforms
val_ds.transform = val_transforms

# Create MONAI DataLoaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=4, pin_memory=torch.cuda.is_available())

Nyul & Udupa normalization will be skipped. Please ensure 'intensity-normalization' is properly installed (e.g., version 1.7.0) and its API matches.
Base Project Directory: C:\Users\punit\Downloads\ISLES-2022\ISLES-2022\isles_data_set
Resolved Raw Data Root: C:\Users\punit\Downloads\ISLES-2022\ISLES-2022\isles_data_set\data
Resolved Raw Mask Root: C:\Users\punit\Downloads\ISLES-2022\ISLES-2022\isles_data_set\data\derivatives
Conceptual Preprocessed Data Root (for offline saving): C:\Users\punit\Downloads\ISLES-2022\ISLES-2022\isles_data_set\data\ISLES-2022_preprocessed
Scanning for 3D samples in raw data root: C:\Users\punit\Downloads\ISLES-2022\ISLES-2022\isles_data_set\data
Total 3D samples found: 248
Total training subjects: 198
Total validation subjects: 50


In [78]:
from monai.networks.nets import UNet
from monai.networks.layers import Norm, Act

class MultiEncoderUNet(nn.Module):
    def __init__(self, in_channels_dwi, in_channels_adc, in_channels_flair, out_channels,
                 spatial_dims=3, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2)):
        super().__init__()
        # Separate encoders for each modality [2, 32, 33, 46]
        self.encoder_dwi = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_dwi,
            out_channels=out_channels, # This output is just before bottleneck
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder # Access the encoder part

        self.encoder_adc = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_adc,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        self.encoder_flair = UNet(
            spatial_dims=spatial_dims,
            in_channels=in_channels_flair,
            out_channels=out_channels,
            channels=channels,
            strides=strides,
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU
        ).encoder

        # Shared decoder (output channels for each encoder are summed at bottleneck)
        # The input channels to the decoder will be sum of bottleneck channels from all encoders
        # For simplicity, let's assume the last channel in 'channels' is the bottleneck feature size
        bottleneck_features = channels[-1] * 3 # Assuming 3 modalities
        self.decoder = UNet(
            spatial_dims=spatial_dims,
            in_channels=bottleneck_features,
            out_channels=out_channels,
            channels=channels[::-1], # Reverse channels for decoder
            strides=strides[::-1], # Reverse strides for decoder
            num_res_units=2,
            norm=Norm.BATCH,
            act=Act.LEAKYRELU,
            is_decoder=True # Indicate this is the decoder part
        ) # This is a simplified representation. A true MultiEncoderUNet would manage skip connections carefully.

    def forward(self, x):
        # x is expected to be a dictionary or a concatenated tensor
        # For this simplified example, assume x is already concatenated
        # In a real MONAI pipeline, you'd pass a dictionary and handle it with transforms
        # For demonstration, let's assume input x has 3 channels for DWI, ADC, FLAIR
        dwi_input = x[:, 0:1, :, :, :] # Assuming channel 0 is DWI
        adc_input = x[:, 1:2, :, :, :] # Assuming channel 1 is ADC
        flair_input = x[:, 2:3, :, :, :] # Assuming channel 2 is FLAIR

        # Encode each modality
        dwi_features = self.encoder_dwi(dwi_input)
        adc_features = self.encoder_adc(adc_input)
        flair_features = self.encoder_flair(flair_input)

        # Concatenate features at bottleneck (simplified, actual nnU-Net handles skip connections)
        # This is a conceptual bottleneck fusion. Actual Multi-encoder nnU-Net combines features
        # from corresponding encoder layers to feed into the decoder's skip connections.
        # For this simplified UNet decoder, we'll just concatenate the deepest features.
        fused_bottleneck = torch.cat([dwi_features, adc_features, flair_features], dim=1) #  for deepest features

        # Pass through shared decoder
        # This is a placeholder. A true UNet decoder requires skip connections from encoders.
        # For a full implementation, consider adapting MONAI's UNet or SegResNet to accept
        # multiple encoder outputs and merge them into the decoder's skip pathways.
        # For now, we'll pass the fused bottleneck into a simple decoder.
        # A more accurate implementation would require custom UNet structure or MONAI's support for multi-input.
        # For this example, let's use a standard UNet and assume the input has 3 channels (DWI, ADC, FLAIR)
        # and the UNet handles the multi-channel input directly as a single input.
        # The "Multi-encoder" part would be handled by the data pipeline preparing the input.
        # The previous section's `ConcatItemsd` already creates a 3-channel input.
        # So, we revert to a standard UNet for simplicity of demonstration,
        # but note that Multi-encoder nnU-Net is conceptually superior.
        pass

# Revert to standard MONAI UNet for demonstration, assuming concatenated input
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = UNet(
    spatial_dims=3,
    in_channels=3, # 3 modalities: DWI, ADC, FLAIR
    out_channels=1, # Binary segmentation: lesion/non-lesion
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
    norm=Norm.BATCH,
    act=Act.LEAKYRELU
).to(device)

# For a true Multi-encoder nnU-Net, one would build a custom network that takes
# a dictionary of images (e.g., {'dwi': tensor, 'adc': tensor, 'flair': tensor})
# and processes them through separate encoders before combining features.
# This is more complex than a simple UNet and would require a custom MONAI network definition.

In [79]:
loss_function = DiceCELoss(to_onehot_y=False, sigmoid=True) # Combines Dice Loss and Cross-Entropy Loss [35, 36, 44]
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)
dice_metric = DiceMetric(include_background=False, reduction="mean") # For evaluation [40]

In [ ]:
max_epochs = 2
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

for epoch in range(max_epochs):
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{max_epochs}"):
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"Epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            val_images = None
            val_labels = None
            val_outputs = None
            for val_data in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}/{max_epochs}"):
                val_inputs, val_labels_batch = val_data["image"].to(device), val_data["label"].to(device)
                val_outputs_batch = model(val_inputs)

                # For DiceMetric, outputs should be one-hot or binary
                val_outputs_batch = (val_outputs_batch.sigmoid() > 0.5).float()

                if val_images is None:
                    val_images = val_inputs
                    val_labels = val_labels_batch
                    val_outputs = val_outputs_batch
                else:
                    val_images = torch.cat([val_images, val_inputs], dim=0)
                    val_labels = torch.cat([val_labels, val_labels_batch], dim=0)
                    val_outputs = torch.cat([val_outputs, val_outputs_batch], dim=0)

            # Calculate Dice Metric
            dice_metric(y_pred=val_outputs, y=val_labels)
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)

            scheduler.step(metric) # Update learning rate based on validation metric

            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join('best_metric_model.pth'))
                print(f"Epoch {epoch + 1} new best metric: {best_metric:.4f} saved model.")
            else:
                print(f"Epoch {epoch + 1} current metric: {metric:.4f} (best: {best_metric:.4f} at epoch {best_metric_epoch})")

print(f"Training complete. Best validation Dice: {best_metric:.4f} at epoch {best_metric_epoch}")

Training Epoch 1/2:   0%|          | 0/99 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load(os.path.join('best_metric_model.pth')))
model.eval()